# ⚡ Tutorial 1: Pre-Training a Sub-Quadratic Kilat LLM From Scratch & Exporting to Hugging Face

Welcome to the official **Kilat** pre-training tutorial! In this interactive notebook, you will learn how to:
1. **Configure a Macro-Synergy Kilat model** combining $O(N)$ Global Exponential Decay (even layers, 0 KV-cache) and Decoupled RoPE Latent MLA (odd layers).
2. **Run the 6-Gate Pre-Flight Assurance Pipeline (`G0`–`G6`)** to mathematically verify initialization, causal masking, and gradient connectivity before spending compute.
3. **Train from scratch** using either:
   - **Option A (Recommended for Pre-Training)**: Kilat's native `KilatTrainer` with decoupled weight decay (`log_lambda` and `RMSNorm` protected) and `FlightRecorderCallback`.
   - **Option B (Hugging Face Ecosystem)**: Standard `AutoConfig.for_model("kilat", ...)` and `AutoModelForCausalLM.from_config(config)` via `register_kilat()`.
4. **Export a standalone Hugging Face Hub bundle (`export_to_hf`)** that anyone can load with `AutoModelForCausalLM.from_pretrained(..., trust_remote_code=True)` without installing `kilat`.

In [ ]:
# Uncomment to install in Google Colab:
# !pip install -q git+https://github.com/Airukua/Kilat.git peft transformers safetensors

## 1. Configure the Macro-Synergy Architecture & Run 6-Gate Pre-Flight Assurance

Before training, we instantiate a compact `KilatConfig` and run `run_verification_pipeline` to prove:
- **Gate 0**: Static security & code hygiene.
- **Gate 1**: Unbiased initial loss $|\mathcal{L}(0) - \ln(V)| \le 0.15$ and bounded residual variance.
- **Gate 2**: Zero future token leakage and 100% trainable parameter gradient connectivity.
- **Gate 3**: Exact equivalence between full prefill and token-by-token `LayerHybridCache` decoding.
- **Gate 4–6**: Data pipeline forensics, single-batch memorization dynamics, and bitwise checkpoint parity.

In [ ]:
import tempfile
import torch
from kilat.configs.model_config import KilatConfig
from kilat.models.transformer import KilatTransformer
from kilat.assurance.runner import run_verification_pipeline

torch.manual_seed(42)

config = KilatConfig(
    vocab_size=256,
    n_embd=64,
    n_head=4,
    n_layer=4,
    max_position_embeddings=64,
    recall_pattern="macro_synergy",  # Even layers: Global Decay | Odd layers: Latent MLA
    ffn_mode="swiglu",
    use_rope=True,
    dropout=0.0,
)

model = KilatTransformer(config)
print(model)

with tempfile.TemporaryDirectory() as tmp_dir:
    report = run_verification_pipeline(config=config, model=model, output_dir=tmp_dir)
    print(f"\nPre-Flight Assurance Status: {report.overall_status}")
    for gate in report.gates:
        print(f"  [{gate.status}] {gate.gate_id} ({gate.duration_ms:.1f} ms)")

## 2. Option A: Pre-Training From Scratch with Native `KilatTrainer`

`KilatTrainer` automatically routes 1D parameters (`RMSNorm`, `bias`, and recurrent `log_lambda` decay horizons) to the `weight_decay=0.0` optimizer group and attaches `FlightRecorderCallback` for real-time NaN/spike telemetry.

In [ ]:
from torch.utils.data import DataLoader
from kilat.data.dataset import SyntheticTokenDataset
from kilat.data.collator import KilatDataCollator
from kilat.engine.args import TrainingArguments
from kilat.engine.trainer import KilatTrainer

train_dataset = SyntheticTokenDataset(vocab_size=config.vocab_size, seq_len=32, num_samples=64, seed=42)
collator = KilatDataCollator(pad_token_id=0, max_length=32)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collator)

work_dir = tempfile.mkdtemp(prefix="kilat_pretrain_")
training_args = TrainingArguments(
    output_dir=work_dir,
    training_mode="steps",
    max_steps=20,
    learning_rate=1e-3,
    warmup_steps=4,
    scheduler_type="cosine",
    logging_steps=5,
    save_checkpoints=True,
    seed=42,
)

trainer = KilatTrainer(model=model, args=training_args, train_dataloader=train_loader)
state = trainer.train()
print(f"Finished {state.global_step} pre-training steps. Final loss: {state.log_history[-1]['loss']:.4f}")

## 3. Option B: Initializing & Training From Scratch via Hugging Face `AutoModelForCausalLM`

By calling `register_kilat()`, you can also construct a fresh Kilat model from scratch using standard Hugging Face `AutoConfig` and `AutoModelForCausalLM.from_config(hf_config)`.

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM
from kilat.integrations.huggingface import register_kilat

register_kilat()

hf_config = AutoConfig.for_model(
    "kilat",
    vocab_size=256,
    n_embd=64,
    n_layer=2,
    n_head=4,
    recall_pattern="macro_synergy",
    ffn_mode="swiglu",
    use_rope=True,
)

# Initialize random weights from scratch using Hugging Face AutoModelForCausalLM!
scratch_hf_model = AutoModelForCausalLM.from_config(hf_config)
sample_ids = torch.randint(0, 256, (2, 16))
out = scratch_hf_model(input_ids=sample_ids, labels=sample_ids)
print("HF From-Scratch Model Class:", type(scratch_hf_model).__name__)
print("Initial Causal LM Loss:", float(out.loss.item()))

## 4. Export a Standalone Hugging Face Bundle (`trust_remote_code=True`)

`export_to_hf()` packages your trained checkpoint, `config.json` (with `auto_map`), and self-contained `configuration_kilat.py` / `modeling_kilat.py` files so downstream users can load your model from Hugging Face Hub without installing `kilat`.

In [ ]:
from pathlib import Path
from kilat.inference.export_hf import convert_checkpoint_to_hf

ckpt_dir = Path(work_dir) / "checkpoint-final"
model.save_pretrained(ckpt_dir)

hf_export_dir = Path(work_dir) / "hf_hub_bundle"
convert_checkpoint_to_hf(checkpoint_dir=ckpt_dir, output_dir=hf_export_dir)

loaded_hub_model = AutoModelForCausalLM.from_pretrained(str(hf_export_dir), trust_remote_code=True)
loaded_hub_model.eval()

prompt = torch.randint(0, 256, (1, 6))
generated = loaded_hub_model.generate(prompt, max_new_tokens=8, do_sample=False)
print("Exported files:", sorted(p.name for p in hf_export_dir.iterdir()))
print("Generated token IDs:", generated.tolist())